In [ ]:
pip install opencv-python tensorflow keras mediapipe numpy

: 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout, LSTM, Reshape

def build_robust_model():
    model = Sequential([
        # Block 1
        Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(48, 48, 1)),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25), # حماية من الحفظ

        # Block 2
        Conv2D(128, (5,5), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25),

        # Block 3
        Conv2D(512, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25),

        Flatten(),

        # تجهيز البيانات للـ LSTM
        Reshape((1, -1)), 
        LSTM(128),
        Dropout(0.5), # حماية قوية قبل الطبقة الأخيرة

        Dense(7, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_robust_model()
model.summary()


In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# تحديد المسارات (تأكد من اسم المجلد عندك في Kaggle)
train_dir = '/kaggle/input/datasets/msambare/fer2013/train' 
test_dir = '/kaggle/input/datasets/msambare/fer2013/test'

batch_size = 64
img_size = 48 # حجم صور FER-2013 الثابت

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# تعديل الـ Generator ليكون أقوى في استخلاص الملامح
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,       # دوران خفيف
    width_shift_range=0.1,   # تحريك عرضي
    height_shift_range=0.1,  # تحريك طولي
    shear_range=0.1,
    zoom_range=0.1,          # زووم خفيف
    horizontal_flip=True,    # قلب الصورة
    fill_mode='nearest'
)

# الـ Test دايماً بيفضل ثابت (Rescale فقط)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    batch_size=32,           # قللنا الـ Batch لـ 32 عشان الموديل يركز أكتر
    color_mode="grayscale",
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(48, 48),
    batch_size=32,
    color_mode="grayscale",
    class_mode='categorical'
)

from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout, LSTM, Reshape

model = Sequential([
    # CNN Layer 1
    Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(48, 48, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    # CNN Layer 2
    Conv2D(128, (5,5), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    # CNN Layer 3
    Conv2D(512, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    Flatten(),

    # إضافة طبقة Dense قبل الـ LSTM لتقليل الأبعاد
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    # تحويل البيانات لشكل يناسب الـ LSTM (Timesteps=1, Features=256)
    Reshape((1, 256)),
    LSTM(128, return_sequences=False),

    # الطبقة النهائية (7 مشاعر)
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout, LSTM, Reshape

model = Sequential([
    # CNN Layer 1
    Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(48, 48, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    # CNN Layer 2
    Conv2D(128, (5,5), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    # CNN Layer 3
    Conv2D(512, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    Flatten(),

    # إضافة طبقة Dense قبل الـ LSTM لتقليل الأبعاد
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    # تحويل البيانات لشكل يناسب الـ LSTM (Timesteps=1, Features=256)
    Reshape((1, 256)),
    LSTM(128, return_sequences=False),

    # الطبقة النهائية (7 مشاعر)
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
import cv2
import numpy as np
import os
from tensorflow.keras.models import load_model

# 1. إعداد المسارات بشكل ذكي (عشان يشتغل في أي مكان)
current_dir = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(current_dir, 'final_emotion_model.h5')

# 2. تعريف تسميات المشاعر (بالترتيب الصحيح لبيانات FER2013)
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

# 3. محاولة تحميل الموديل
if not os.path.exists(model_path):
    print(f"❌ خطأ: لم يتم العثور على الموديل في: {model_path}")
    print("تأكد من وجود ملف final_emotion_model.h5 بجانب هذا الكود.")
    exit()

print("⏳ جاري تحميل الموديل... استعد")
try:
    model = load_model(model_path)
    print("✅ الموديل جاهز! سيتم فتح الكاميرا الآن.")
except Exception as e:
    print(f"❌ حدث خطأ أثناء تحميل الموديل: {e}")
    exit()

# 4. تحميل مصنف الوجوه (Haar Cascade)
face_classifier = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 5. تشغيل الكاميرا
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ فشل في التقاط الصورة من الكاميرا")
        break

    # تحويل الصورة لرمادي (التصحيح النهائي للـ Error اللي قابلك)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # اكتشاف الوجوه في الصورة
    faces = face_classifier.detectMultiScale(gray_frame, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        # رسم مستطيل حول الوجه المكتشف
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        
        # استقطاع منطقة الوجه ومعالجتها (Preprocessing)
        roi_gray = gray_frame[y:y+h, x:x+w]
        roi_gray = cv2.resize(roi_gray, (48, 48), interpolation=cv2.INTER_AREA)
        
        # تجهيز البيانات للموديل (Normalization)
        roi = roi_gray.astype('float') / 255.0
        roi = np.expand_dims(roi, axis=0)      # إضافة Batch dimension
        roi = np.expand_dims(roi, axis=-1)     # إضافة Channel dimension (Grayscale)

        # التنبؤ بالشعور (استخدام verbose=0 عشان الـ CMD يفضل نظيف)
        prediction = model.predict(roi, verbose=0)[0]
        label = emotion_labels[prediction.argmax()]
        
        # كتابة النتيجة فوق المستطيل
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # عرض النافذة النهائية
    cv2.imshow('Real-time Emotion Detector', frame)

    # اضغط حرف 'q' للخروج
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 6. إغلاق الكاميرا والنوافذ
cap.release()
cv2.destroyAllWindows()
print("👋 تم إغلاق البرنامج بنجاح.")